## Data Exploration

In [1]:
# loading the corpus to memory

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from utils import RAW_DATA_DIR

with open(RAW_DATA_DIR / "crime_and_punishment.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Total characters: {len(text):,}")
print(f"First 300 characters:\n{text[:300]}")

Total characters: 1,135,110
First 300 characters:
CRIME AND PUNISHMENT

By Fyodor Dostoevsky



Translated By Constance Garnett




TRANSLATOR’S PREFACE

A few words about Dostoevsky himself may help the English reader to
understand his work.

Dostoevsky was the son of a doctor. His parents were very hard-working
and deeply religious people, but so


In [2]:
# locating where the actual novel starts

idx = text.find("PART I")
print(f"'PART I' first found at character index: {idx}")
print(text[idx:idx+300])

'PART I' first found at character index: 4584
PART I



CHAPTER I

On an exceptionally hot evening early in July a young man came out of
the garret in which he lodged in S. Place and walked slowly, as though
in hesitation, towards K. bridge.

He had successfully avoided meeting his landlady on the staircase. His
garret was under the roof of a h


In [3]:
# trimming everything before the novel's actual start

text = text[idx:]

print(f"Trimmed length: {len(text):,} characters")
print(f"New start:\n{text[:200]}")

Trimmed length: 1,130,526 characters
New start:
PART I



CHAPTER I

On an exceptionally hot evening early in July a young man came out of
the garret in which he lodged in S. Place and walked slowly, as though
in hesitation, towards K. bridge.

He 


In [4]:
# inspecting the last 500 characters to check for trailing footnotes which are not needed

print(text[-500:])

e seven years as though they were seven days. He did not
know that the new life would not be given him for nothing, that he would
have to pay dearly for it, that it would cost him great striving, great
suffering.

But that is the beginning of a new story--the story of the gradual
renewal of a man, the story of his gradual regeneration, of his passing
from one world into another, of his initiation into a new unknown life.
That might be the subject of a new story, but our present story is
ended.




In [5]:
# saving the trimmed, novel-only text

from utils import PROCESSED_DATA_DIR
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

with open(PROCESSED_DATA_DIR / "corpus_clean.txt", "w", encoding="utf-8") as f:
    f.write(text)

print(f"Saved {len(text):,} characters to {PROCESSED_DATA_DIR / 'corpus_clean.txt'}")

Saved 1,130,526 characters to C:\Users\NASA2004\OneDrive\Documents\rnn-lstm-long-range-memory\data\processed\corpus_clean.txt


In [6]:
# merging Raskolnikov's nickname "Rodya" into "Raskolnikov"

import re
text = re.sub(r"\bRodya\b", "Raskolnikov", text)

# re-saving the corpus with the merge applied
with open(PROCESSED_DATA_DIR / "corpus_clean.txt", "w", encoding="utf-8") as f:
    f.write(text)

print(f"'Rodya' occurrences remaining: {len(re.findall(r'Rodya', text))}")
print(f"'Raskolnikov' occurrences now: {len(re.findall(r'Raskolnikov', text))}")

'Rodya' occurrences remaining: 0
'Raskolnikov' occurrences now: 911


In [7]:
# counting how often each unique character appears in the corpus

from collections import Counter

char_counts = Counter(text)
sorted_counts = char_counts.most_common()

print(f"Total unique characters: {len(char_counts)}")
print(f"\nTop 10 most common:")
for ch, count in sorted_counts[:10]:
    print(f"  {repr(ch)}: {count:,}")

print(f"\nBottom 20 rarest:")
for ch, count in sorted_counts[-20:]:
    print(f"  {repr(ch)}: {count}")

Total unique characters: 91

Top 10 most common:
  ' ': 184,929
  'e': 102,558
  't': 77,274
  'a': 70,794
  'o': 70,270
  'n': 61,371
  'i': 55,680
  'h': 53,363
  's': 50,936
  'r': 45,391

Bottom 20 rarest:
  '[': 6
  ']': 6
  'ô': 4
  'æ': 4
  '1': 3
  '4': 3
  'ê': 3
  'ä': 3
  '8': 2
  '7': 2
  'à': 2
  '6': 1
  '3': 1
  'î': 1
  '9': 1
  'ç': 1
  'ö': 1
  'ü': 1
  'è': 1
  '5': 1


In [8]:
# checking characters just above the rarest-20 cutoff to find a clean threshold boundary

for ch, count in sorted_counts[-35:-20]:
    print(f"  {repr(ch)}: {count}")

  'F': 249
  ':': 233
  'C': 231
  'ï': 222
  'E': 200
  'Z': 194
  '(': 156
  ')': 156
  'V': 91
  'J': 59
  'U': 46
  '*': 31
  'Q': 21
  'é': 17
  'X': 7


In [9]:
# finding every "PART" and "CHAPTER" heading and its position in the text

import re

part_matches = [(m.start(), m.group()) for m in re.finditer(r"PART [IVX]+", text)]
chapter_matches = [(m.start(), m.group()) for m in re.finditer(r"CHAPTER [IVX]+", text)]

print(f"Found {len(part_matches)} PART markers:")
for pos, label in part_matches:
    print(f"  {label} at char {pos:,}")

print(f"\nFound {len(chapter_matches)} CHAPTER markers (first 10 shown):")
for pos, label in chapter_matches[:10]:
    print(f"  {label} at char {pos:,}")

Found 6 PART markers:
  PART I at char 0
  PART II at char 192,483
  PART III at char 408,777
  PART IV at char 579,720
  PART V at char 738,658
  PART VI at char 898,211

Found 39 CHAPTER markers (first 10 shown):
  CHAPTER I at char 10
  CHAPTER II at char 18,286
  CHAPTER III at char 58,143
  CHAPTER IV at char 87,988
  CHAPTER V at char 115,783
  CHAPTER VI at char 138,768
  CHAPTER VII at char 166,260
  CHAPTER I at char 192,494
  CHAPTER II at char 229,852
  CHAPTER III at char 252,740


In [10]:
# scanning for capitalized word frequency to find candidate character names

capitalized_words = re.findall(r"\b[A-Z][a-z]+\b", text)
name_candidates = Counter(capitalized_words)

print("Top 30 most frequent capitalized words:")
for word, count in name_candidates.most_common(30):
    print(f"  {word}: {count}")

Top 30 most frequent capitalized words:
  He: 1249
  Raskolnikov: 911
  And: 747
  But: 683
  You: 590
  The: 535
  It: 513
  What: 477
  She: 442
  Sonia: 402
  Razumihin: 347
  Dounia: 325
  Ivanovna: 304
  Petrovitch: 287
  Why: 244
  That: 224
  Katerina: 216
  Yes: 206
  Porfiry: 206
  There: 204
  They: 198
  No: 190
  Well: 182
  Pyotr: 173
  Oh: 146
  In: 144
  How: 141
  If: 138
  At: 137
  Pulcheria: 123


In [11]:
# importing the vocab-building functions from src/data.py

from data import build_vocab, encode

train_text = text[:738058]
val_text = text[738058:897611]
test_text = text[897611:]

char2idx, idx2char = build_vocab(train_text, min_count=10)
print(f"Vocab size: {len(idx2char)}")

val_encoded = encode(val_text, char2idx)
test_encoded = encode(test_text, char2idx)

unk_idx = char2idx["<UNK>"]
val_unk_count = val_encoded.count(unk_idx)
test_unk_count = test_encoded.count(unk_idx)

print(f"Val <UNK> count: {val_unk_count} / {len(val_encoded)}")
print(f"Test <UNK> count: {test_unk_count} / {len(test_encoded)}")

Vocab size: 71
Val <UNK> count: 9 / 159553
Test <UNK> count: 22 / 233677


In [12]:
# testing chunking on train split

from data import make_chunks, CharDataset
from torch.utils.data import DataLoader

train_encoded = encode(train_text, char2idx)
train_chunks = make_chunks(train_encoded, seq_len=100)

print(f"Number of train chunks: {len(train_chunks)}")

train_dataset = CharDataset(train_chunks)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

sample_input, sample_target = next(iter(train_loader))
print(f"Batch input shape: {sample_input.shape}")
print(f"Batch target shape: {sample_target.shape}")

Number of train chunks: 7380
Batch input shape: torch.Size([64, 100])
Batch target shape: torch.Size([64, 100])


In [13]:
# anchor position tracking

from data import find_anchor_positions, ANCHOR_NAMES

train_anchor_positions = find_anchor_positions(train_text)

for name in ANCHOR_NAMES:
    print(f"{name}: {len(train_anchor_positions[name])} occurrences")

Raskolnikov: 668 occurrences
Sonia: 130 occurrences
Razumihin: 296 occurrences
Dounia: 216 occurrences
Katerina: 80 occurrences
Porfiry: 153 occurrences


In [14]:
# checking rnn shapes
from rnn import VanillaRNN

vocab_size = len(idx2char)
model = VanillaRNN(vocab_size=vocab_size, embed_dim=64, hidden_size=256)

sample_input, sample_target = next(iter(train_loader))

logits, h_final, hidden_states = model(sample_input, track_gradients=True)

print(f"Input shape:        {sample_input.shape}")
print(f"Logits shape:       {logits.shape}")
print(f"Final hidden shape: {h_final.shape}")
print(f"Hidden states tracked: {len(hidden_states)}")

Input shape:        torch.Size([64, 100])
Logits shape:       torch.Size([64, 100, 71])
Final hidden shape: torch.Size([64, 256])
Hidden states tracked: 100


In [15]:
# checking lstm shapes
from lstm import LSTMModel

lstm_model = LSTMModel(vocab_size=vocab_size, embed_dim=64, hidden_size=256)

logits, h_final, c_final, hidden_states = lstm_model(sample_input, track_gradients=True)

print(f"Input shape:        {sample_input.shape}")
print(f"Logits shape:       {logits.shape}")
print(f"Final hidden shape: {h_final.shape}")
print(f"Final cell shape:   {c_final.shape}")
print(f"Hidden states tracked: {len(hidden_states)}")

Input shape:        torch.Size([64, 100])
Logits shape:       torch.Size([64, 100, 71])
Final hidden shape: torch.Size([64, 256])
Final cell shape:   torch.Size([64, 256])
Hidden states tracked: 100


In [16]:
# smoke test with two epochs
import torch
from rnn import VanillaRNN
from lstm import LSTMModel
from train import train
from data import make_chunks, CharDataset
from torch.utils.data import DataLoader

device = torch.device("cpu")
vocab_size = len(idx2char)

val_encoded = encode(val_text, char2idx)
val_chunks = make_chunks(val_encoded, seq_len=100)
val_dataset = CharDataset(val_chunks)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, drop_last=True)

# RNN smoke test
rnn_model = VanillaRNN(vocab_size=vocab_size, embed_dim=64, hidden_size=256)
rnn_train_losses, rnn_val_losses = train(
    rnn_model, train_loader, val_loader,
    num_epochs=2, lr=0.001, device=device,
    model_type="rnn", save_path="checkpoints/rnn_best.pt"
)

# LSTM smoke test
lstm_model = LSTMModel(vocab_size=vocab_size, embed_dim=64, hidden_size=256)
lstm_train_losses, lstm_val_losses = train(
    lstm_model, train_loader, val_loader,
    num_epochs=2, lr=0.001, device=device,
    model_type="lstm", save_path="checkpoints/lstm_best.pt"
)

Epoch 1/2 | Train Loss: 2.5410 | Val Loss: 2.1928 | Train Perplexity: 12.69 | Val Perplexity: 8.96
  → Saved best checkpoint (val loss: 2.1928)
Epoch 2/2 | Train Loss: 2.0145 | Val Loss: 1.9420 | Train Perplexity: 7.50 | Val Perplexity: 6.97
  → Saved best checkpoint (val loss: 1.9420)
Epoch 1/2 | Train Loss: 2.7587 | Val Loss: 2.3605 | Train Perplexity: 15.78 | Val Perplexity: 10.60
  → Saved best checkpoint (val loss: 2.3605)
Epoch 2/2 | Train Loss: 2.1752 | Val Loss: 2.0911 | Train Perplexity: 8.80 | Val Perplexity: 8.09
  → Saved best checkpoint (val loss: 2.0911)
